# 🧬 AI-Assisted Protein Function Classifier

**Machine Learning Pipeline for Broad Protein Functional Classification**

This notebook implements a scientifically rigorous, reproducible, and transparent machine learning pipeline to predict broad protein functional classes from primary amino-acid sequences.


## 1. Project Overview & 2. Install Dependencies


In [ ]:
!pip install biopython scikit-learn pandas numpy matplotlib seaborn joblib requests -q
print("✅ Dependencies verified successfully!")


## 3. Feature Extraction Module (33 Sequence-Derived Features)

The feature extraction engine calculates 33 sequence-derived properties for each protein sequence:


In [ ]:
import numpy as np
import pandas as pd
from typing import Dict, List, Optional, Tuple

AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")

AA_MOLECULAR_WEIGHTS = {
    'A': 89.09, 'C': 121.15, 'D': 133.10, 'E': 147.13, 'F': 165.19,
    'G': 75.07, 'H': 155.16, 'I': 131.17, 'K': 146.19, 'L': 131.17,
    'M': 149.21, 'N': 132.12, 'P': 115.13, 'Q': 146.15, 'R': 174.20,
    'S': 105.09, 'T': 119.12, 'V': 117.15, 'W': 204.23, 'Y': 181.19
}

HYDROPATHY = {
    'A': 1.8, 'C': 2.5, 'D': -3.5, 'E': -3.5, 'F': 2.8,
    'G': -0.4, 'H': -3.2, 'I': 4.5, 'K': -3.9, 'L': 3.8,
    'M': 1.9, 'N': -3.5, 'P': -1.6, 'Q': -3.5, 'R': -4.5,
    'S': -0.8, 'T': -0.7, 'V': 4.2, 'W': -0.9, 'Y': -1.3
}

PK_VALUES = {
    'N_term': 9.69, 'C_term': 2.34,
    'D': 3.86, 'E': 4.25, 'C': 8.33, 'Y': 10.07,
    'H': 6.00, 'K': 10.54, 'R': 12.48
}

def clean_sequence(sequence: str) -> str:
    """Strips whitespace/newlines and converts amino acid sequence to uppercase."""
    if not sequence or not isinstance(sequence, str):
        return ""
    sequence = sequence.upper().strip()
    sequence = ''.join(sequence.split())
    valid_sequence = ''.join([aa for aa in sequence if aa in AMINO_ACIDS])
    return valid_sequence

def validate_sequence(sequence: str) -> Tuple[bool, str, List[str]]:
    """Validates sequence length and detects invalid amino acid characters without silent deletion."""
    if not sequence or not isinstance(sequence, str):
        return False, "Empty or invalid sequence input.", []
    cleaned = ''.join(sequence.upper().split())
    if not cleaned:
        return False, "Sequence contains no characters.", []
    invalid_chars = sorted(list(set(c for c in cleaned if c not in AMINO_ACIDS)))
    if invalid_chars:
        return False, f"Invalid amino acid characters detected: {', '.join(invalid_chars)}", invalid_chars
    if len(cleaned) < 50:
        return False, f"Sequence too short ({len(cleaned)} AA). Minimum 50 amino acids required.", []
    return True, "", []

def amino_acid_composition(sequence: str) -> Dict[str, float]:
    """Calculates normalized amino acid frequencies."""
    length = len(sequence)
    if length == 0:
        return {aa: 0.0 for aa in AMINO_ACIDS}
    return {aa: sequence.count(aa) / length for aa in AMINO_ACIDS}

def molecular_weight(sequence: str) -> float:
    """Calculates approximate protein molecular weight in Daltons with water loss subtraction."""
    if len(sequence) == 0:
        return 0.0
    water_weight = 18.015
    mw = sum(AA_MOLECULAR_WEIGHTS.get(aa, 0.0) for aa in sequence)
    mw -= (len(sequence) - 1) * water_weight
    return mw

def _charge_at_pH(sequence: str, pH: float) -> float:
    """Calculates net protein charge at specified pH."""
    positive = 1.0 / (1.0 + 10**(pH - PK_VALUES['N_term']))
    negative = 1.0 / (1.0 + 10**(PK_VALUES['C_term'] - pH))
    for aa in sequence:
        if aa in ['K', 'R', 'H']:
            positive += 1.0 / (1.0 + 10**(pH - PK_VALUES[aa]))
        elif aa in ['D', 'E', 'C', 'Y']:
            negative += 1.0 / (1.0 + 10**(PK_VALUES[aa] - pH))
    return positive - negative

def isoelectric_point(sequence: str) -> float:
    """Estimates protein isoelectric point (pI) using binary search charge titration."""
    if len(sequence) == 0:
        return 7.0
    pH_min, pH_max = 0.0, 14.0
    for _ in range(100):
        pH_mid = (pH_min + pH_max) / 2.0
        charge = _charge_at_pH(sequence, pH_mid)
        if abs(charge) < 0.001:
            return pH_mid
        if charge > 0:
            pH_min = pH_mid
        else:
            pH_max = pH_mid
    return (pH_min + pH_max) / 2.0

def gravy_score(sequence: str) -> float:
    """Calculates Grand Average of Hydropathy (GRAVY) score."""
    if len(sequence) == 0:
        return 0.0
    return sum(HYDROPATHY.get(aa, 0.0) for aa in sequence) / len(sequence)

def hydrophobic_segment_count(sequence: str, min_length: int = 5) -> int:
    """Counts continuous hydrophobic residue stretches (>= 5 residues)."""
    hydrophobic_aas = set(['A', 'V', 'I', 'L', 'M', 'F', 'W', 'P'])
    count = 0
    current_len = 0
    for aa in sequence:
        if aa in hydrophobic_aas:
            current_len += 1
        else:
            if current_len >= min_length:
                count += 1
            current_len = 0
    if current_len >= min_length:
        count += 1
    return count

def extract_features(sequence: str) -> Optional[np.ndarray]:
    """Extracts complete 33-feature numerical vector from amino acid sequence."""
    seq = clean_sequence(sequence)
    if len(seq) < 50:
        return None
    features = []
    # 1. 20 Amino acid composition features
    aa_comp = amino_acid_composition(seq)
    for aa in AMINO_ACIDS:
        features.append(aa_comp[aa])
    # 2. Sequence length
    features.append(float(len(seq)))
    # 3. Molecular weight
    features.append(molecular_weight(seq))
    # 4. Isoelectric point
    features.append(isoelectric_point(seq))
    # 5. GRAVY score
    features.append(gravy_score(seq))
    
    # 6. Biological residue group fractions
    length = float(len(seq))
    features.append(sum(seq.count(aa) for aa in ['F', 'W', 'Y']) / length)        # aromatic_fraction
    features.append(sum(seq.count(aa) for aa in ['D', 'E', 'K', 'R', 'H']) / length)  # charged_fraction
    features.append(sum(seq.count(aa) for aa in ['A', 'V', 'I', 'L', 'M', 'F', 'W', 'P']) / length)  # hydrophobic_fraction
    features.append(sum(seq.count(aa) for aa in ['S', 'T', 'N', 'Q', 'Y', 'C']) / length)  # polar_fraction
    features.append(sum(seq.count(aa) for aa in ['K', 'R', 'H']) / length)        # basic_fraction
    features.append(sum(seq.count(aa) for aa in ['D', 'E']) / length)            # acidic_fraction
    features.append(seq.count('C') / length)                                      # cysteine_fraction
    features.append(seq.count('P') / length)                                      # proline_fraction
    features.append(float(hydrophobic_segment_count(seq)))                        # hydrophobic_segments
    
    return np.array(features)

def get_feature_names() -> List[str]:
    """Returns ordered feature names for visualization and interpretability."""
    names = [f"AA_{aa}" for aa in AMINO_ACIDS]
    names.extend([
        "seq_length", "mol_weight", "isoelectric_point", "gravy_score",
        "aromatic_fraction", "charged_fraction", "hydrophobic_fraction", "polar_fraction",
        "basic_fraction", "acidic_fraction", "cysteine_fraction", "proline_fraction", "hydrophobic_segments"
    ])
    return names

print("✅ Feature extraction module loaded successfully! (33 sequence features)")


## 4. UniProt Dataset Collection & High-Quality Labeling



In [ ]:
import requests
import time
import re

UNIPROT_API = "https://rest.uniprot.org/uniprotkb/search"

CLASS_QUERIES = {
    "Enzyme": '(keyword:"Hydrolase" OR keyword:"Oxidoreductase" OR keyword:"Transferase" OR keyword:"Kinase" OR keyword:"Lyase" OR keyword:"Isomerase" OR keyword:"Ligase") AND (reviewed:true) AND (length:[100 TO 1000])',
    "Transporter": '(keyword:"Transport" OR keyword:"Ion transport" OR keyword:"Ion channel") AND NOT (keyword:"Hydrolase" OR keyword:"Oxidoreductase" OR keyword:"Transferase" OR keyword:"Kinase") AND (reviewed:true) AND (length:[100 TO 1000])',
    "Binding": '(keyword:"DNA-binding" OR keyword:"RNA-binding" OR keyword:"Receptor") AND NOT (keyword:"Hydrolase" OR keyword:"Oxidoreductase" OR keyword:"Transferase" OR keyword:"Kinase" OR keyword:"Transport" OR keyword:"Transcription regulation") AND (reviewed:true) AND (length:[100 TO 1000])',
    "Regulatory": '(keyword:"Transcription regulation" OR keyword:"Transcription factor" OR keyword:"Repressor") AND NOT (keyword:"Hydrolase" OR keyword:"Oxidoreductase" OR keyword:"Transferase" OR keyword:"Kinase" OR keyword:"Transport") AND (reviewed:true) AND (length:[100 TO 1000])'
}

TARGET_PER_CLASS = 280

def fetch_proteins_for_class(class_name: str, query: str, target_count: int = 280) -> Tuple[list, int]:
    proteins = []
    removed_ambiguous = 0
    page_size = 100
    next_url = UNIPROT_API
    params = {
        "query": query,
        "format": "json",
        "fields": "accession,sequence,keyword,protein_name,cc_function",
        "size": page_size
    }
    
    print(f"  Fetching '{class_name}' proteins...")
    while len(proteins) < target_count and next_url:
        try:
            if next_url == UNIPROT_API:
                response = requests.get(next_url, params=params, timeout=30)
            else:
                response = requests.get(next_url, timeout=30)
            response.raise_for_status()
            data = response.json()
            results = data.get("results", [])
            if not results:
                break
                
            for entry in results:
                accession = entry.get("primaryAccession", "")
                seq = entry.get("sequence", {}).get("value", "")
                
                function_text = ""
                if "comments" in entry:
                    for comment in entry.get("comments", []):
                        if comment.get("commentType") == "FUNCTION":
                            texts = comment.get("texts", [])
                            if texts:
                                function_text = texts[0].get("value", "")
                                
                protein_name = entry.get("proteinDescription", {}).get("recommendedName", {}).get("fullName", {}).get("value", "")
                
                skip_terms = ["hypothetical", "uncharacterized", "putative", "probable", "unknown function", "fragment"]
                combined_text = (protein_name + " " + function_text).lower()
                if any(term in combined_text for term in skip_terms):
                    removed_ambiguous += 1
                    continue
                    
                keywords_list = [kw.get("name", "") for kw in entry.get("keywords", [])]
                
                protein = {
                    "accession": accession,
                    "sequence": seq,
                    "protein_name": protein_name,
                    "keywords": "; ".join(keywords_list),
                    "label": class_name
                }
                
                if seq and len(seq) >= 50:
                    proteins.append(protein)
                    if len(proteins) >= target_count:
                        break
                        
            link_header = response.headers.get("Link", "")
            next_url = None
            if link_header:
                match = re.search(r'<([^>]+)>; rel="next"', link_header)
                if match:
                    next_url = match.group(1)
            time.sleep(0.3)
        except requests.RequestException as e:
            print(f"    Warning: UniProt request issue for '{class_name}': {e}")
            break
            
    print(f"  → Fetched {len(proteins)} reviewed samples for {class_name} (filtered {removed_ambiguous} ambiguous entries)")
    return proteins, removed_ambiguous

def fetch_all_proteins() -> Tuple[pd.DataFrame, int]:
    all_proteins = []
    total_ambiguous_removed = 0
    print("=" * 50)
    print("UniProt Data Fetcher (Curated & Reviewed)")
    print("=" * 50)
    
    for class_name, query in CLASS_QUERIES.items():
        print(f"\n[{class_name}]")
        proteins, amb_count = fetch_proteins_for_class(class_name, query, TARGET_PER_CLASS)
        all_proteins.extend(proteins)
        total_ambiguous_removed += amb_count
        
    return pd.DataFrame(all_proteins), total_ambiguous_removed

print("⏳ Downloading protein data from UniProt REST API...")
raw_protein_df, total_ambiguous_removed = fetch_all_proteins()
print("\n✅ Data collection complete!")


## 5. Dataset Quality Analysis & Data Leakage Prevention

In [ ]:
orig_records = len(raw_protein_df)

# Accession deduplication
df_acc_uniq = raw_protein_df.drop_duplicates(subset=["accession"])
uniq_accessions = len(df_acc_uniq)
removed_dup_accessions = orig_records - uniq_accessions

# Sequence deduplication
protein_df = df_acc_uniq.drop_duplicates(subset=["sequence"]).copy().reset_index(drop=True)
uniq_sequences = len(protein_df)
removed_dup_sequences = uniq_accessions - uniq_sequences

print("Dataset Quality Analysis")
print("-------------------------")
print(f"Initial records:                      {orig_records}")
print(f"Removed duplicate accessions:         {removed_dup_accessions}")
print(f"Removed duplicate sequences:          {removed_dup_sequences}")
print(f"Removed unclear/ambiguous proteins:   {total_ambiguous_removed}")
print(f"Final dataset samples:                {len(protein_df)}")
print(f"Unique accessions count:              {uniq_accessions}")
print(f"Unique sequences count:               {uniq_sequences}")


## 6. Class Distribution & Balance Visualization


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
class_counts = protein_df["label"].value_counts()
colors = ['#2ecc71', '#3498db', '#e74c3c', '#9b59b6']
ax = sns.barplot(x=class_counts.index, y=class_counts.values, hue=class_counts.index, palette=colors, legend=False)

plt.title('Protein Functional Class Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Functional Class', fontsize=12)
plt.ylabel('Number of Sequences', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)

total_samples = len(protein_df)
for p in ax.patches:
    height = int(p.get_height())
    percentage = (height / total_samples) * 100
    ax.annotate(f"{height} ({percentage:.1f}%)", (p.get_x() + p.get_width() / 2., height),
                ha='center', va='bottom', fontsize=11, fontweight='bold', xytext=(0, 3), textcoords='offset points')

plt.tight_layout()
plt.show()

print("\nClass Distribution Breakdown:")
dist_df = pd.DataFrame({
    'Count': class_counts,
    'Percentage': (class_counts / total_samples * 100).round(2).astype(str) + '%'
})
print(dist_df.to_string())


## 7. Feature Preparation


In [ ]:
print("Extracting features for all dataset sequences...")
X_list = []
y_list = []

for idx, row in protein_df.iterrows():
    features = extract_features(row["sequence"])
    if features is not None:
        X_list.append(features)
        y_list.append(row["label"])

X = np.vstack(X_list)
y = np.array(y_list)

print(f"✅ Feature matrix constructed! X shape: {X.shape}, y shape: {y.shape}")


## 8. Stratified Train/Test Split (80/20)


In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"Training samples: {len(X_train)} ({len(X_train)/len(X):.0%})")
print(f"Test samples:     {len(X_test)} ({len(X_test)/len(X):.0%})")


## 9. 5-Fold Stratified Cross-Validation (Training Set Only)

We evaluate model performance stability using **5-Fold Stratified Cross-Validation** strictly inside the training set.


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_model = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)

cv_accs, cv_macro_f1s, cv_weighted_f1s = [], [], []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]
    
    cv_model.fit(X_tr, y_tr)
    preds = cv_model.predict(X_val)
    
    acc = accuracy_score(y_val, preds)
    _, _, mac_f1, _ = precision_recall_fscore_support(y_val, preds, average='macro', zero_division=0)
    _, _, w_f1, _ = precision_recall_fscore_support(y_val, preds, average='weighted', zero_division=0)
    
    cv_accs.append(acc)
    cv_macro_f1s.append(mac_f1)
    cv_weighted_f1s.append(w_f1)

print("5-Fold Cross Validation")
print("-----------------------")
print(f"Accuracy :    {np.mean(cv_accs):.2%} ± {np.std(cv_accs):.2%}")
print(f"Macro F1 :    {np.mean(cv_macro_f1s):.4f} ± {np.std(cv_macro_f1s):.4f}")
print(f"Weighted F1 : {np.mean(cv_weighted_f1s):.4f} ± {np.std(cv_weighted_f1s):.4f}")


## 10. Random Forest Model Training

We fit our primary **Random Forest Classifier** on the entire training set (`n_estimators=300`, `class_weight="balanced"`, `random_state=42`).


In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("✅ Primary Random Forest model trained successfully!")


## 11. Model Comparison Against Lightweight Baselines

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import balanced_accuracy_score

models = {
    "Stratified Dummy Baseline": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    "SVM (RBF Kernel)": make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, class_weight="balanced", probability=True, random_state=RANDOM_STATE)),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
}

comp_results = []
dummy_acc = 0.0

for name, clf in models.items():
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)
    bal_acc = balanced_accuracy_score(y_test, preds)
    _, _, mac_f1, _ = precision_recall_fscore_support(y_test, preds, average='macro', zero_division=0)
    _, _, w_f1, _ = precision_recall_fscore_support(y_test, preds, average='weighted', zero_division=0)
    
    if name == "Stratified Dummy Baseline":
        dummy_acc = acc
        
    comp_results.append({
        "Model": name,
        "Accuracy": f"{acc:.2%}",
        "Balanced Accuracy": f"{bal_acc:.2%}",
        "Macro F1": f"{mac_f1:.4f}",
        "Weighted F1": f"{w_f1:.4f}"
    })

comparison_df = pd.DataFrame(comp_results)
print("=" * 70)
print("MODEL COMPARISON TABLE (HELD-OUT TEST SET)")
print("=" * 70)
print(comparison_df.to_string(index=False))
print("=" * 70)

rf_acc = accuracy_score(y_test, model.predict(X_test))
abs_imp = (rf_acc - dummy_acc) * 100
print(f"\n📊 Stratified dummy baseline accuracy: {dummy_acc:.2%}")
print(f"🎯 Random Forest accuracy:            {rf_acc:.2%}")
print(f"📈 Absolute improvement:             +{abs_imp:.2f} percentage points over baseline")


## 12. Final Test Set Evaluation & Metrics


In [ ]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
y_probas = model.predict_proba(X_test)

test_acc = accuracy_score(y_test, y_pred)
bal_acc = balanced_accuracy_score(y_test, y_pred)
prec_mac, rec_mac, f1_mac, _ = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)
prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)

print(f"🎯 Final Test Accuracy:    {test_acc:.2%}")
print(f"⚖️ Balanced Accuracy:      {bal_acc:.2%}")
print(f"📊 Macro F1 Score:         {f1_mac:.4f}")
print(f"📈 Weighted F1 Score:      {f1_w:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

classes = model.classes_
p_cls, r_cls, f1_cls, supp_cls = precision_recall_fscore_support(y_test, y_pred, labels=classes, zero_division=0)

print("=" * 50)
print("Class-wise Performance Analysis")
print("=" * 50)
class_f1s = {}
for i, cls in enumerate(classes):
    print(f"\n{cls}:")
    print(f"  Precision: {p_cls[i]:.4f}")
    print(f"  Recall:    {r_cls[i]:.4f}")
    print(f"  F1:        {f1_cls[i]:.4f}")
    print(f"  Support:   {supp_cls[i]}")
    class_f1s[cls] = f1_cls[i]

best_class = max(class_f1s, key=class_f1s.get)
weakest_class = min(class_f1s, key=class_f1s.get)
print("-" * 50)
print(f"🏆 Best performing class:    {best_class} (F1: {class_f1s[best_class]:.4f})")
print(f"⚠️ Weakest performing class: {weakest_class} (F1: {class_f1s[weakest_class]:.4f})")


## 13. Raw & Normalized Confusion Matrices


In [ ]:
from sklearn.metrics import confusion_matrix

cm_raw = confusion_matrix(y_test, y_pred, labels=classes)
cm_norm = confusion_matrix(y_test, y_pred, labels=classes, normalize='true')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(cm_raw, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, ax=axes[0], cbar=False)
axes[0].set_title('Raw Confusion Matrix (Counts)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontsize=11)
axes[0].set_ylabel('True Label', fontsize=11)

sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', xticklabels=classes, yticklabels=classes, ax=axes[1], cbar=False)
axes[1].set_title('Normalized Confusion Matrix (Percentages)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted Label', fontsize=11)
axes[1].set_ylabel('True Label', fontsize=11)

plt.tight_layout()
plt.show()


## 14. Multiclass ROC-AUC Analysis (One-vs-Rest)


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.preprocessing import label_binarize

macro_roc_auc = roc_auc_score(y_test, y_probas, multi_class='ovr', average='macro')
weighted_roc_auc = roc_auc_score(y_test, y_probas, multi_class='ovr', average='weighted')

print(f"🌐 Multiclass Macro ROC-AUC:    {macro_roc_auc:.4f}")
print(f"⚖️ Multiclass Weighted ROC-AUC: {weighted_roc_auc:.4f}")

y_test_bin = label_binarize(y_test, classes=classes)
n_classes = len(classes)

plt.figure(figsize=(8, 6))
colors_roc = ['#2ecc71', '#3498db', '#e74c3c', '#9b59b6']

for i, color in zip(range(n_classes), colors_roc):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_probas[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{classes[i]} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Chance')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=11)
plt.ylabel('True Positive Rate', fontsize=11)
plt.title('Multiclass One-vs-Rest (OvR) ROC Curves', fontsize=13, fontweight='bold')
plt.legend(loc="lower right", fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 15. Feature Importance & Model Interpretability

In [ ]:
from sklearn.inspection import permutation_importance

feature_names = get_feature_names()
importances = model.feature_importances_
top_indices = np.argsort(importances)[::-1][:15]

plt.figure(figsize=(10, 6))
plt.barh(range(15), importances[top_indices][::-1], color='#3498db')
plt.yticks(range(15), [feature_names[i] for i in top_indices][::-1])
plt.xlabel('Random Forest Impurity Importance (MDI)', fontsize=11)
plt.title('Top 15 Most Important Features (Split Utility)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Top 15 Most Important Features (RF Impurity Table):")
print("Rank | Feature                  | Importance")
print("-" * 45)
for rank, idx in enumerate(top_indices, 1):
    print(f" {rank:2d}  | {feature_names[idx]:24s} | {importances[idx]:.4f}")

perm_imp = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE)
top_perm_indices = np.argsort(perm_imp.importances_mean)[::-1][:15]

print("\nTop 15 Features by Permutation Importance (Test Set Impact):")
print("Rank | Feature                  | Mean Importance ± Std")
print("-" * 55)
for rank, idx in enumerate(top_perm_indices, 1):
    print(f" {rank:2d}  | {feature_names[idx]:24s} | {perm_imp.importances_mean[idx]:.4f} ± {perm_imp.importances_std[idx]:.4f}")


## 16. Sequence Prediction Module

In [ ]:
def predict_function(sequence: str) -> dict:
    is_valid, err_msg, invalid_chars = validate_sequence(sequence)
    if not is_valid:
        return {
            "predicted_class": None,
            "confidence": {},
            "top_probability": 0.0,
            "confidence_category": "N/A",
            "sequence_length": len(sequence) if sequence else 0,
            "error": err_msg
        }
        
    seq = clean_sequence(sequence)
    features = extract_features(seq)
    if features is None:
        return {
            "predicted_class": None,
            "confidence": {},
            "top_probability": 0.0,
            "confidence_category": "N/A",
            "sequence_length": len(seq),
            "error": "Feature extraction failed."
        }
        
    X_in = features.reshape(1, -1)
    pred_cls = model.predict(X_in)[0]
    probs = model.predict_proba(X_in)[0]
    
    conf_dict = {cls: round(float(p), 4) for cls, p in zip(model.classes_, probs)}
    conf_dict = dict(sorted(conf_dict.items(), key=lambda item: item[1], reverse=True))
    top_prob = max(probs)
    
    if top_prob >= 0.70:
        conf_cat = "High"
    elif top_prob >= 0.50:
        conf_cat = "Moderate"
    else:
        conf_cat = "Low"
        
    return {
        "predicted_class": pred_cls,
        "confidence": conf_dict,
        "top_probability": round(float(top_prob), 4),
        "confidence_category": conf_cat,
        "sequence_length": len(seq),
        "error": None
    }

def format_result(protein_name: str, result: dict) -> str:
    if result.get("error"):
        return f"❌ Prediction Failed for {protein_name}:\n  {result['error']}"
    
    output = [
        "=" * 50,
        f"🧬 Protein: {protein_name}",
        f"Sequence Length: {result['sequence_length']} aa",
        f"\nPredicted Class: {result['predicted_class']}",
        f"Prediction Confidence: {result['confidence_category']} ({result['top_probability']:.2%})",
        "\nClass Probabilities:",
        "-" * 30
    ]
    for cls, prob in result["confidence"].items():
        bar = "■" * int(prob * 20)
        output.append(f"  {cls:12s}: {prob:.2%} {bar}")
    output.append("=" * 50)
    return "\n".join(output)

print("✅ Prediction function ready!")


## 17. Live Prediction: Human Insulin


In [ ]:
insulin_seq = "MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN"

result = predict_function(insulin_seq)
print(format_result("Human Insulin", result))


## 18. Live Prediction: Hemoglobin (Model Limitations Test)

Hemoglobin is an oxygen-binding protein. We test whether primary sequence composition features correctly capture this function or exhibit classification limitations.


In [ ]:
hemoglobin_seq = "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR"

result = predict_function(hemoglobin_seq)
print(format_result("Human Hemoglobin", result))

print("\nNote:")
print("This example demonstrates that sequence-composition features alone")
print("may not reliably distinguish all biological functions.")


## 19. Live Prediction: Aquaporin-1 (Transporter Test)


In [ ]:
aquaporin_seq = "MASEFKKKLFWRAVVAEFLATTLFVFISIGSALGFKYPVGNNQTAVQDNVKVSLAFGLSIATLAQSVGHISGAHLNPAVTLGLLLSCQISIFRALPDDRIGGANGIPLGSLCDTGATSFGHAGILSLTLAIHISGIVAGLITGSALPEVGPAAILAVALVHGTTLGLGRMAIGTIASVGALWDEAVWIGFPIGLGLALAVFYVNLLISGALKQDIAAGFLGPNHTTVGVAMVVPMITLCAVNLSRHYFTIAFYTMAIAGIAGGILSLGLVATHLKAGISSGAAFHINPAITLGIGTFGNIQVVFNKFNNWTFSGILGYGSASLMNPVLIPLATTLFAFAGKAFNLNAKQIISGPASGCGMENGGVEIGFHSPGIMKAGWIGIYFLPGAIFYLGPSVAHQ"

result = predict_function(aquaporin_seq)
print(format_result("Human Aquaporin-1", result))


## 20. Edge Case Testing


In [ ]:
print("--- Test Case 1: Short Sequence (< 50 AA) ---")
short_result = predict_function("ABC")
print(format_result("Short Peptide", short_result))

print("\n--- Test Case 2: Invalid Amino Acid Characters ---")
invalid_result = predict_function("ACDEFG123XYZ")
print(format_result("Corrupted Sequence", invalid_result))


## 21. Model Interpretation


In [ ]:
# Calculate most confused pair dynamically from raw confusion matrix
np.fill_diagonal(cm_raw, 0)
max_conf_idx = np.unravel_index(np.argmax(cm_raw, axis=None), cm_raw.shape)
true_conf_class = classes[max_conf_idx[0]]
pred_conf_class = classes[max_conf_idx[1]]
most_important_feat = feature_names[top_indices[0]]

print("Model Interpretation Summary")
print("----------------------------")
print(f"1. Strongest performing class:  {best_class} (F1-score: {class_f1s[best_class]:.4f})")
print(f"2. Weakest performing class:    {weakest_class} (F1-score: {class_f1s[weakest_class]:.4f})")
print(f"3. Most frequent confusion:     True '{true_conf_class}' misclassified as '{pred_conf_class}' ({cm_raw[max_conf_idx]} samples)")
print(f"4. Top predictive feature:      {most_important_feat} (RF Impurity: {importances[top_indices[0]]:.4f})")
print("")
print("Biological Context:")
print("Sequence-derived physicochemical features (such as hydropathy and aromatic/charged residue fractions)")
print("provide strong statistical signal for distinguishing broad functional classes. However, functional overlap")
print("and structural nuances create inherent misclassification boundaries for sequence-only ML models.")


## 22. Final Project Summary & Conclusion


In [ ]:
print("==================================================")
print("FINAL MODEL SUMMARY")
print("==================================================")
print(f"Dataset size:               {len(protein_df)}")
print(f"Classes:                    {len(np.unique(y))}")
print(f"Features:                   {X.shape[1]}")
print("")
print(f"Training samples:           {len(X_train)}")
print(f"Test samples:               {len(X_test)}")
print("")
print(f"Cross-validation accuracy:  {np.mean(cv_accs):.2%} ± {np.std(cv_accs):.2%}")
print(f"Test accuracy:              {test_acc:.2%}")
print(f"Balanced accuracy:          {bal_acc:.2%}")
print(f"Macro F1:                   {f1_mac:.4f}")
print(f"Weighted F1:                {f1_w:.4f}")
print(f"Macro ROC-AUC:              {macro_roc_auc:.4f}")
print("")
print(f"Best performing class:      {best_class}")
print(f"Weakest performing class:   {weakest_class}")
print("")
print("Model:                      Random Forest Classifier")
print("==================================================")
